In [1]:
import geopandas as gpd
import requests
import pandas as pd
from pathlib import Path
from shapely.geometry import Polygon, MultiPolygon

RAW_DIR   = Path("../../data/raw/04_red_natura2000")
DELIM_DIR = Path("../../data/raw/delimitations")
RAW_DIR.mkdir(parents=True, exist_ok=True)

GEOJSON_HUESCA = DELIM_DIR / "Huesca_Delimitacion.geojson"
OUTPUT_GPKG    = RAW_DIR / "red_natura_huesca.gpkg"

huesca = gpd.read_file(GEOJSON_HUESCA).to_crs("EPSG:25830")

BASE = "https://bio.discomap.eea.europa.eu/arcgis/rest/services/ProtectedSites/Natura2000Query_WM/MapServer/1/query"

def rings_to_shape(rings):
    """Convierte formato ArcGIS rings a geometría Shapely."""
    if not rings:
        return None
    polys = [Polygon(ring) for ring in rings if len(ring) >= 3]
    if not polys:
        return None
    return MultiPolygon(polys) if len(polys) > 1 else polys[0]

todos_attrs = []
todos_geoms = []
offset = 0

while True:
    params = {
        "where":             "MS='ES'",
        "outFields":         "SITECODE,SITENAME,SITETYPE,MS",
        "returnGeometry":    "true",
        "outSR":             "4326",
        "f":                 "json",
        "resultRecordCount": 500,
        "resultOffset":      offset
    }

    r = requests.get(BASE, params=params, timeout=120)
    data = r.json()

    if "error" in data:
        print(f"Error en offset {offset}:", data["error"]["message"])
        break

    features = data.get("features", [])
    if not features:
        break

    for f in features:
        geom = rings_to_shape(f.get("geometry", {}).get("rings", []))
        todos_geoms.append(geom)
        todos_attrs.append(f.get("attributes", {}))

    print(f"Descargadas: {len(todos_attrs)} features...")

    if len(features) < 500:
        break
    offset += 500

print(f"\nTotal España: {len(todos_attrs)}")

# Construir GeoDataFrame
natura_es = gpd.GeoDataFrame(
    pd.DataFrame(todos_attrs),
    geometry=todos_geoms,
    crs="EPSG:4326"
).to_crs("EPSG:25830")

# Clip a Huesca
natura_huesca = gpd.clip(natura_es, huesca)
natura_huesca = natura_huesca[~natura_huesca.geometry.is_empty].reset_index(drop=True)

print(f"Espacios en Huesca: {len(natura_huesca)}")
print("Tipos (SITETYPE):", natura_huesca["SITETYPE"].value_counts().to_dict())
print(f"Cobertura: {natura_huesca.geometry.union_all().area / 1e6:.1f} km²")

natura_huesca.to_file(OUTPUT_GPKG, driver="GPKG")
print(f"\nGuardado: {OUTPUT_GPKG}")


Descargadas: 500 features...


Descargadas: 1000 features...


Descargadas: 1500 features...


Descargadas: 1807 features...

Total España: 1807


Espacios en Huesca: 102
Tipos (SITETYPE): {'B': 67, 'A': 24, 'C': 11}


Cobertura: 4940.2 km²

Guardado: ../../data/raw/04_red_natura2000/red_natura_huesca.gpkg


In [2]:
import geopandas as gpd
import pydeck as pdk
from shapely.ops import unary_union
import pandas as pd
from pathlib import Path

RAW_DIR   = Path("../../data/raw/04_red_natura2000")
DELIM_DIR = Path("../../data/raw/delimitations")
MAP_DIR   = Path("../../data/map/04_red_natura2000")
MAP_DIR.mkdir(parents=True, exist_ok=True)

# --- Cargar capas ---
provincia = gpd.read_file(DELIM_DIR / "Huesca_Delimitacion.geojson").to_crs("EPSG:4326")
natura = gpd.read_file(RAW_DIR / "red_natura_huesca.gpkg").to_crs("EPSG:4326")

# --- Separar por tipo ---
zec  = natura[natura["SITETYPE"] == "B"].copy()
zepa = natura[natura["SITETYPE"] == "A"].copy()
ambas = natura[natura["SITETYPE"] == "C"].copy()

# --- Disolver geometrías por tipo ---
geom_zec  = unary_union(zec.geometry)
geom_zepa = unary_union(zepa.geometry)
geom_ambas = unary_union(ambas.geometry)

# --- Calcular solapamientos reales ---
# Zonas que son ZEC y ZEPA simultáneamente
solapamiento = geom_zec.intersection(geom_zepa)

# ZEC puro (sin solapamiento con ZEPA)
zec_puro  = geom_zec.difference(geom_zepa)

# ZEPA puro (sin solapamiento con ZEC)
zepa_puro = geom_zepa.difference(geom_zec)

# Combinar solapamiento con los que ya vienen como tipo C
if not solapamiento.is_empty and not geom_ambas.is_empty:
    ambas_final = solapamiento.union(geom_ambas)
elif not solapamiento.is_empty:
    ambas_final = solapamiento
elif not geom_ambas.is_empty:
    ambas_final = geom_ambas
else:
    ambas_final = None

# --- Construir features para pydeck ---
color_map = {
    "ZEC":  [34,  139,  34, 160],
    "ZEPA": [30,  144, 255, 160],
    "C":    [255, 140,   0, 190],
}

def geom_to_features(geom, tipo, nombre):
    if geom is None or geom.is_empty:
        return []
    features = []
    # Aplanar cualquier tipo de colección
    if geom.geom_type == "Polygon":
        polys = [geom]
    elif geom.geom_type == "MultiPolygon":
        polys = list(geom.geoms)
    elif geom.geom_type == "GeometryCollection":
        # Filtrar solo polígonos, descartar puntos y líneas residuales
        polys = [g for g in geom.geoms if g.geom_type in ("Polygon", "MultiPolygon")]
        # Aplanar MultiPolygon dentro de la colección
        polys_flat = []
        for g in polys:
            if g.geom_type == "Polygon":
                polys_flat.append(g)
            else:
                polys_flat.extend(list(g.geoms))
        polys = polys_flat
    else:
        return []

    color = color_map[tipo]
    for p in polys:
        if p.is_empty or len(list(p.exterior.coords)) < 3:
            continue
        features.append({
            "type": "Feature",
            "geometry": {"type": "Polygon", "coordinates": [list(p.exterior.coords)]},
            "properties": {"tipo": nombre, "color": color}
        })
    return features

natura_features = (
    geom_to_features(zec_puro,    "ZEC",  "ZEC (Directiva Hábitats)")  +
    geom_to_features(zepa_puro,   "ZEPA", "ZEPA (Directiva Aves)")     +
    geom_to_features(ambas_final, "C",    "ZEC + ZEPA")
)

geojson_natura = {"type": "FeatureCollection", "features": natura_features}

# --- Límite provincia ---
boundary_features = []
for geom in provincia.geometry:
    polys = [geom] if geom.geom_type == "Polygon" else list(geom.geoms)
    for p in polys:
        boundary_features.append({
            "type": "Feature",
            "geometry": {"type": "Polygon", "coordinates": [list(p.exterior.coords)]},
            "properties": {}
        })

geojson_boundary = {"type": "FeatureCollection", "features": boundary_features}

# --- Layers ---
layer_natura = pdk.Layer(
    "GeoJsonLayer",
    data=geojson_natura,
    stroked=True,
    filled=True,
    get_fill_color="properties.color",
    get_line_color=[255, 255, 255, 80],
    get_line_width=100,
    line_width_min_pixels=1,
    pickable=True,
)

layer_boundary = pdk.Layer(
    "GeoJsonLayer",
    data=geojson_boundary,
    stroked=True,
    filled=False,
    get_line_color=[255, 255, 255, 200],
    get_line_width=800,
    line_width_min_pixels=2,
)

# --- Vista y deck ---
view = pdk.ViewState(
    longitude=-0.08, latitude=41.95,
    zoom=8, min_zoom=7, max_zoom=13,
    pitch=0, bearing=0,
)

tooltip = {
    "html": "<b>{tipo}</b>",
    "style": {"backgroundColor": "#111", "color": "white",
              "fontSize": "12px", "padding": "6px"},
}

deck = pdk.Deck(
    layers=[layer_natura, layer_boundary],
    initial_view_state=view,
    tooltip=tooltip,
    map_style="https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
)

leyenda_html = """
<div style="position:absolute; bottom:30px; left:20px; background:rgba(0,0,0,0.75);
            padding:14px 18px; border-radius:8px; font-family:sans-serif;
            font-size:13px; color:white; z-index:999; line-height:1.9;">
  <div style="font-weight:bold; font-size:14px; margin-bottom:8px;"> Red Natura 2000 — Huesca</div>
  <div><span style="display:inline-block;width:14px;height:14px;background:#228b22;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>ZEC (Directiva Hábitats)</div>
  <div><span style="display:inline-block;width:14px;height:14px;background:#1e90ff;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>ZEPA (Directiva Aves)</div>
  <div><span style="display:inline-block;width:14px;height:14px;background:#ff8c00;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>ZEC + ZEPA</div>
</div>
"""

html_output = deck.to_html(as_string=True)
html_output = html_output.replace("</body>", leyenda_html + "</body>")

output_html = MAP_DIR / "mapa_red_natura.html"
with open(output_html, "w", encoding="utf-8") as f:
    f.write(html_output)

print(f"Guardado: {output_html}")

Guardado: ../../data/map/04_red_natura2000/mapa_red_natura.html
